In [1]:
# %%
#!uv pip install torch==2.6.0 torchvision==0.21.0 numpy==2.2.4 matplotlib==3.10.1 tqdm==4.66.2

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from tqdm.auto import tqdm

from steganoGAN.utils import bits_to_bytearray

In [4]:
# Check pytorch version and device
print(f'pytorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

pytorch: 2.6.0+cu126
CUDA: True


# Prepare dataset

In [9]:
from torchvision import datasets, transforms
from torchvision.transforms.v2 import functional as TF, RandomCrop, RandomHorizontalFlip, ColorJitter, Compose, ToTensor
from torch.utils.data import DataLoader
from PIL import Image
import os

_DEFAULT_MU = [.5, .5, .5]
_DEFAULT_SIGMA = [.5, .5, .5]
DEFAULT_TRANSFORM = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(360, pad_if_needed=True),
    transforms.ToTensor(),
    transforms.Normalize(_DEFAULT_MU, _DEFAULT_SIGMA),
])


In [10]:
import os
import requests
from zipfile import ZipFile

def download_div2k(output_dir="./div2k_dataset"):
    """
    Downloads and extracts the DIV2K dataset into the specified directory.
    Skips download if ZIP files already exist.
    Skips extraction if directories already exist and contain files.

    Args:
        output_dir (str): Path to the directory where the dataset should be stored.
    """
    # URLs for DIV2K dataset files
    urls = {
        "DIV2K_train_HR": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip",
        "DIV2K_train_LR_bicubic_X4": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X4.zip",
        "DIV2K_valid_HR": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip",
        "DIV2K_valid_LR_bicubic_X4": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X4.zip"
    }

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for name, url in urls.items():
        # File paths
        zip_path = os.path.join(output_dir, f"{name}.zip")
        extract_path = os.path.join(output_dir, name)

        # Check if ZIP file needs to be downloaded
        if not os.path.exists(zip_path):
            print(f"Downloading {name}...")
            response = requests.get(url, stream=True)
            with open(zip_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024):
                    f.write(chunk)
            print(f"{name} downloaded successfully!")
        else:
            print(f"ZIP file for {name} already exists. Skipping download.")

        # Check if extraction directory exists and has content
        if os.path.exists(extract_path) and os.listdir(extract_path):
            print(f"Data for {name} already extracted. Skipping extraction.")
        else:
            print(f"Extracting {name}...")
            os.makedirs(extract_path, exist_ok=True)
            with ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_path)
            print(f"{name} extracted to {extract_path}.")

    print("DIV2K dataset download and extraction completed!")

# Call the function to download the dataset
download_div2k()


ZIP file for DIV2K_train_HR already exists. Skipping download.
Data for DIV2K_train_HR already extracted. Skipping extraction.
ZIP file for DIV2K_train_LR_bicubic_X4 already exists. Skipping download.
Data for DIV2K_train_LR_bicubic_X4 already extracted. Skipping extraction.
ZIP file for DIV2K_valid_HR already exists. Skipping download.
Data for DIV2K_valid_HR already extracted. Skipping extraction.
ZIP file for DIV2K_valid_LR_bicubic_X4 already exists. Skipping download.
Data for DIV2K_valid_LR_bicubic_X4 already extracted. Skipping extraction.
DIV2K dataset download and extraction completed!


In [14]:
class Div2KDataset(Dataset):
    def __init__(self, hr_dir, lr_dir, transform=None):
        self.hr_images = sorted(os.listdir(hr_dir))
        self.lr_images = sorted(os.listdir(lr_dir))
        self.hr_dir = hr_dir
        self.lr_dir = lr_dir
        self.transform = transform

    def __len__(self):
        return len(self.hr_images)

    def __getitem__(self, idx):
        hr_path = os.path.join(self.hr_dir, self.hr_images[idx])
        lr_path = os.path.join(self.lr_dir, self.lr_images[idx])

        hr_image = Image.open(hr_path).convert("RGB")
        lr_image = Image.open(lr_path).convert("RGB")

        if self.transform:
            hr_image = self.transform(hr_image)
            lr_image = self.transform(lr_image)

        return hr_image, lr_image

In [15]:
# Define paths to HR and LR directories
hr_dir = "./div2k_dataset/DIV2K_train_HR/DIV2K_train_HR"
lr_dir = "./div2k_dataset/DIV2K_train_LR_bicubic_X4/DIV2K_train_LR_bicubic/X4"
val_hr_dir = "./div2k_dataset/DIV2K_valid_HR/DIV2K_valid_HR"
val_lr_dir = "./div2k_dataset/DIV2K_valid_LR_bicubic_X4/DIV2K_valid_LR_bicubic/X4"


# Define transforms (e.g., ToTensor)
transform = DEFAULT_TRANSFORM
def custom_collate(batch):
    # Extract just the first element (hr_image) from each item in the batch
    images = torch.stack([item[0] for item in batch])
    # Return in the format expected by SteganoGAN
    return images, None

# Create dataset and dataloader
train_dataset = Div2KDataset(hr_dir=hr_dir, lr_dir=lr_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,collate_fn=custom_collate)
val_dataset = Div2KDataset(hr_dir=val_hr_dir, lr_dir=val_lr_dir, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate)


# Training process

In [8]:
from steganoGAN.steganogan import SteganoGAN
from steganoGAN import utils, encoder, decoder,critic
steganogan = SteganoGAN(1, encoder.SteganographyEncoder, decoder.SteganographyDecoder, critic.SteganographyDiscriminator, hidden_size=32, use_cuda=True, verbose=True)
steganogan.fit(train_loader, val_loader, epochs=10)

Using cuda device
Moving components to cuda...
Epoch 1/10


100%|██████████| 25/25 [00:15<00:00,  1.65it/s]


Epoch 2/10


100%|██████████| 25/25 [00:14<00:00,  1.71it/s]


Epoch 3/10


...

100%|██████████| 25/25 [00:15<00:00,  1.59it/s]


Epoch 9/10


100%|██████████| 25/25 [00:15<00:00,  1.57it/s]


Epoch 10/10


100%|██████████| 25/25 [00:16<00:00,  1.47it/s]


In [9]:
steganogan.save('pretrained/basic_10epochs.steg')

In [10]:
from steganoGAN.steganogan import SteganoGAN
steganogan = SteganoGAN.load(path='pretrained/basic_10_epochs.steg')

Moving components to cuda...


In [36]:
from steganoGAN.steganogan import SteganoGAN
model = SteganoGAN.load(path='pretrained/basic_10_epochs.steg')

Moving components to cuda...


In [41]:
model.encode('./covers/cover.jpg','./covers/stegano10.png','This is a super secret message!')

In [42]:
model.decode('./covers/stegano10.png')

ValueError: No valid message found in the image.

In [39]:
def continue_training(pretrained_dir='pretrained', additional_epochs=10, train_loader=None, val_loader=None):
    """
    Load the latest pretrained SteganoGAN model and continue training.

    Args:
        pretrained_dir (str): Directory containing pretrained models
        additional_epochs (int): Number of additional epochs to train
        train_loader: DataLoader with training images
        val_loader: DataLoader with validation images

    Returns:
        Trained SteganoGAN model
    """
    import os
    import glob
    from steganoGAN.steganogan import SteganoGAN
    from steganoGAN import encoder, decoder, critic
    import torch

    # Find all .steg model files in the directory
    model_files = glob.glob(os.path.join(pretrained_dir, '*.steg'))

    if not model_files:
        raise ValueError(f"No .steg model files found in {pretrained_dir}")

    # Sort by modification time (most recent last)
    latest_model_path = max(model_files, key=os.path.getmtime)
    current_epochs = int(latest_model_path.split('_')[1])
    print(f"Loading latest model: {latest_model_path}")

    # First, determine device availability
    model = SteganoGAN(1, encoder.SteganographyEncoder, decoder.SteganographyDecoder, critic.SteganographyDiscriminator, hidden_size=32, use_cuda=True, verbose=True)
    model.completed_epochs = current_epochs
    # Load the model
    model.load(path=latest_model_path)

    # Check if training data is provided
    if train_loader is None or val_loader is None:
        raise ValueError("Both train_loader and val_loader must be provided for continued training")

    print(f"Continuing training for {additional_epochs} more epochs")

    # Continue training the model
    model.fit(train_loader, val_loader, epochs=additional_epochs)

    # Save the updated model with epoch count
    current_epochs = model.completed_epochs
    new_model_path = os.path.join(pretrained_dir, f"basic_{current_epochs}_epochs.steg")
    model.save(new_model_path)
    print(f"Saved updated model to {new_model_path}")

    return model

In [40]:
model = continue_training(
    pretrained_dir='pretrained',
    additional_epochs=10,
    train_loader=train_loader,
    val_loader=val_loader
)

Loading latest model: pretrained\basic_23_epochs.steg
Using cuda device
Moving components to cuda...
Moving components to cuda...
Continuing training for 10 more epochs
Epoch 24/33


100%|██████████| 25/25 [00:14<00:00,  1.75it/s]


Epoch 25/33


100%|██████████| 25/25 [00:14<00:00,  1.74it/s]


Epoch 26/33


...

Epoch 32/33


100%|██████████| 25/25 [00:13<00:00,  1.84it/s]


Epoch 33/33


100%|██████████| 25/25 [00:13<00:00,  1.83it/s]

Saved updated model to pretrained\basic_33_epochs.steg


In [43]:
model.encode('./covers/cover.jpg',f'./covers/stegano_{model.completed_epochs}.png','This is a super secret message')

Message encoded successfully.


In [44]:
model.decode(f'./covers/stegano_{model.completed_epochs}.png')

ValueError: No valid message found in the image.